In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("🚀 A iniciar a fase de Data Preparation...")

# 1. Carregar o dataset original
file_path = '../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# 2. Limpeza básica identificada na EDA
# Remover metadados e colunas com variância zero
cols_to_drop = [
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp',
    'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 
    'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'
]

# Filtrar colunas que realmente existem no df
cols_to_drop_existing = [col for col in cols_to_drop if col in df.columns]
df_clean = df.drop(columns=cols_to_drop_existing)

# Tratar infinitos e nulos
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates()

# Mapear Target (BENIGN -> 0, DDoS -> 1)
df_clean['Label_Num'] = df_clean['Label'].map({'BENIGN': 0, 'DDoS': 1})

print(f"✅ Dataset preparado com {df_clean.shape[0]} linhas e {df_clean.shape[1]} colunas!")

🚀 A iniciar a fase de Data Preparation...
✅ Dataset preparado com 221264 linhas e 70 colunas!


In [ ]:
# 1. Separar as características (X) da variável alvo (y)
X = df_clean.drop(columns=['Label', 'Label_Num'])
y = df_clean['Label_Num']

# 2. Divisão em 80% Treino e 20% Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("--- Divisão Concluída ---")
print(f"• Treino (X_train): {X_train.shape[0]} amostras, {X_train.shape[1]} features")
print(f"• Teste  (X_test):  {X_test.shape[0]} amostras, {X_test.shape[1]} features")
print("\nProporção das classes no treino (0 = BENIGN, 1 = DDoS):")
print(y_train.value_counts(normalize=True).round(4))

In [ ]:
# Criar e aplicar o normalizador
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Normalização concluída com sucesso!")
print(f"• Média aproximada das features em treino: {X_train_scaled.mean():.2f}")
print(f"• Desvio padrão médio em treino: {X_train_scaled.std():.2f}")